# Tool-Router Ladder: Stage G2, step-level GRPO from the SFT adapter

`src/train_grpo.py`: for each train-split state, sample a group of completions from the SFT LoRA, score
each with `src/reward.py` (execution on a replayed tau2 retail DB, no API quota), and push the policy
toward the completions that beat their group. Reference = the SFT adapter, with a KL penalty.

**Settings:** Accelerator **GPU T4 x2**, Internet **on**, the **`tool-router-grpo-states`** dataset attached
(upload `data/grpo/` from the Mac: `states.jsonl` + `manifest.json`, built by `make_pairs.py states`).
**Secrets:** `HF_TOKEN` (write) and `WANDB_API_KEY`, both ticked.
**Run:** Save Version -> **Save & Run All (Commit)**. A checkpoint goes to the Hub every 25 steps; a rerun resumes.
0.5B first (plan X7 kill gate: a rising reward curve before 1.5B). Set `MODEL` below.

**Send back:** the smoke cell's last lines, the W&B link, and the final `[train_grpo] done: ...` line.


In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN", "WANDB_API_KEY"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN", "WANDB_API_KEY"])


In [ ]:
# Step out of the working dir before deleting anything under it (a re-run would
# otherwise leave the kernel in a dead directory).
%cd /kaggle/working
import shutil
shutil.rmtree("/kaggle/working/tau2-bench", ignore_errors=True)
shutil.rmtree("/kaggle/working/the_llm_project", ignore_errors=True)
# tau2 is the reward's executor (make_pairs.Tau2Runner): the retail tools and DB, run locally.
!git clone -q https://github.com/sierra-research/tau2-bench.git /kaggle/working/tau2-bench
!cd /kaggle/working/tau2-bench && git checkout -q 672227c6b6676edc20d57ea53b7000262aae77b9 && pip install -q -e .
!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

!pip install -q peft
# As in kaggle_dpo: Kaggle's torchao 0.10.0 makes peft's LoRA dispatcher raise; nothing here uses it.
!pip uninstall -y -q torchao torchaudio


In [ ]:
# Finds the attached states dataset (make_pairs.py states output: states.jsonl + manifest.json).
import glob, json, os
hits = [h for h in glob.glob("/kaggle/input/**/manifest.json", recursive=True)
        if json.load(open(h)).get("file") == "states.jsonl"]
assert hits, "attach the tool-router-grpo-states dataset (Add Input): no states manifest under /kaggle/input"
STATES = os.path.dirname(hits[0])
m = json.load(open(hits[0]))
print("STATES =", STATES, "| states:", m["n_states"], "reply states:", m["n_reply_states"])
HF_USER = "madhusiddharths1"
MODEL = "qwen05b"   # plan X7: 0.5B first; qwen15b only after 0.5B's reward rises


### 1. Smoke: one step on the longest prompt, 2 samples of 32 tokens (memory and plumbing; no Hub push)


In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/train_grpo.py --config configs/v2/{MODEL}-grpo.yaml --data "$STATES" --parent "$HF_USER/tool-router-{MODEL}-sft" --output-dir /kaggle/working/grpo-smoke --smoke


### 2. The run (300 steps, 4 states x 4 samples per step)


In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/train_grpo.py --config configs/v2/{MODEL}-grpo.yaml --data "$STATES" --parent "$HF_USER/tool-router-{MODEL}-sft" --output-dir /kaggle/working/{MODEL}-grpo --hub-repo "$HF_USER/tool-router-{MODEL}-grpo" --wandb
